# Milestone 2: Up Next & Play History
**Team Zyntra**

In this milestone, we extended the TuneWave player with two critical features:
1. **Up Next Queue (FIFO):** Songs queued by the user play **before** the playlist continues.
2. **Play History Stack (LIFO):** Keeps the last 10 songs played and drops the oldest. Calling `back()` replays the top song and removes it from history.
3. **Core Rule:** Playing from Up Next or history **does not move the playlist position**.


In [1]:
from collections import deque

class Song:
    """Song representation."""
    def __init__(self, song_id, title, artist, duration=0):
        self.song_id = str(song_id)
        self.title = str(title)
        self.artist = str(artist)
        self.duration = duration

    def __repr__(self):
        return f"Song({self.song_id}, '{self.title}' - {self.artist})"

    def __eq__(self, other):
        if isinstance(other, Song):
            return self.song_id == other.song_id
        return self.song_id == str(other)


class SongNode:
    def __init__(self, song: Song):
        self.song = song
        self.prev = None
        self.next = None


class Playlist:
    """Doubly Linked List + Hash Map from Milestone 1."""
    def __init__(self, name="My Playlist"):
        self.name = name
        self.head = None
        self.tail = None
        self.current = None
        self.repeat = False
        self.song_map = {}
        self._count = 0

    def __len__(self):
        return self._count

    def addSong(self, song):
        if not isinstance(song, Song):
            if isinstance(song, dict):
                song = Song(song.get("song_id") or song.get("id"), song.get("title", "Untitled"), song.get("artist", "Unknown"))
            elif isinstance(song, str):
                song = Song(song, song, "Unknown")
            else:
                return False

        if song.song_id in self.song_map:
            return False

        node = SongNode(song)
        if self.head is None:
            self.head = node
            self.tail = node
            self.current = node
        else:
            self.tail.next = node
            node.prev = self.tail
            self.tail = node

        self.song_map[song.song_id] = node
        self._count += 1
        return True

    def removeSong(self, songId):
        sid = str(songId)
        node = self.song_map.get(sid)
        if not node:
            return False

        if node == self.current:
            if node.next is not None:
                self.current = node.next
            elif node.prev is not None:
                self.current = node.prev
            else:
                self.current = None

        if node == self.head:
            self.head = node.next
            if self.head:
                self.head.prev = None
        if node == self.tail:
            self.tail = node.prev
            if self.tail:
                self.tail.next = None

        if node.prev and node != self.head:
            node.prev.next = node.next
        if node.next and node != self.tail:
            node.next.prev = node.prev

        del self.song_map[sid]
        self._count -= 1
        node.prev = None
        node.next = None
        return True

    def next(self):
        if self.current is None:
            return "end"
        if self.current.next is not None:
            self.current = self.current.next
            return self.current.song
        if self.repeat:
            self.current = self.head
            return self.current.song if self.current else "end"
        return "end"

    def previous(self):
        if self.current is None:
            return None
        if self.current.prev is not None:
            self.current = self.current.prev
            return self.current.song
        if self.repeat:
            self.current = self.tail
            return self.current.song if self.current else None
        return self.current.song

    def toggleRepeat(self):
        self.repeat = not self.repeat
        return self.repeat

    def getCurrentSong(self):
        return self.current.song if self.current else None


# -------------------------------------------------------------
# MILESTONE 2: QUEUE & STACK IMPLEMENTATIONS
# -------------------------------------------------------------

class UpNextQueue:
    """FIFO Queue for Up Next songs."""
    def __init__(self):
        self._q = deque()

    def enqueue(self, song: Song):
        self._q.append(song)

    def dequeue(self):
        return self._q.popleft() if self._q else None

    def is_empty(self):
        return len(self._q) == 0

    def __len__(self):
        return len(self._q)

    def to_list(self):
        return list(self._q)


class PlayHistory:
    """
    Bounded LIFO Stack for recently played songs.
    Rule: Keeps last 10 songs played and drops the oldest.
    """
    def __init__(self, capacity=10):
        self.capacity = capacity
        # using maxlen automatically drops oldest item from left when full!
        self._stack = deque(maxlen=capacity)

    def push(self, song: Song):
        self._stack.append(song)

    def pop(self):
        return self._stack.pop() if self._stack else None

    def peek(self):
        return self._stack[-1] if self._stack else None

    def is_empty(self):
        return len(self._stack) == 0

    def __len__(self):
        return len(self._stack)

    def to_list(self):
        return list(self._stack)


class TuneWavePlayer:
    """
    Coordinates Playlist, Up Next Queue, and Play History.
    """
    def __init__(self, playlist=None):
        self.playlist = playlist if playlist is not None else Playlist()
        self.up_next = UpNextQueue()
        self.history = PlayHistory(capacity=10)
        self.catalog = {}  # for songs not yet in playlist
        
        # sync playlist songs into catalog
        if playlist:
            for sid, node in playlist.song_map.items():
                self.catalog[sid] = node.song

        self.currently_playing = self.playlist.getCurrentSong()

    def addToUpNext(self, songId):
        """Adds a song to the Up Next queue."""
        song = None
        if isinstance(songId, Song):
            song = songId
        else:
            sid = str(songId)
            song = self.catalog.get(sid)
            if not song and self.playlist and sid in self.playlist.song_map:
                song = self.playlist.song_map[sid].song

        if not song:
            return False

        self.up_next.enqueue(song)
        return True

    def playNext(self):
        """
        Rules:
        1. Up Next plays before playlist.
        2. Playing from Up Next does NOT move playlist position.
        3. History tracks the song that was playing previously.
        """
        # Check Up Next queue first
        if not self.up_next.is_empty():
            song = self.up_next.dequeue()
            if self.currently_playing is not None:
                self.history.push(self.currently_playing)
            self.currently_playing = song
            # IMPORTANT: playlist position is not moved!
            return self.currently_playing

        # If Up Next is empty, resume the sequential playlist
        if not self.playlist or len(self.playlist) == 0:
            return "end"

        if self.currently_playing is None:
            self.currently_playing = self.playlist.getCurrentSong()
            return self.currently_playing

        next_song = self.playlist.next()
        if next_song == "end":
            return "end"

        # Record previously playing song into history
        self.history.push(self.currently_playing)
        self.currently_playing = next_song
        return self.currently_playing

    def back(self):
        """
        Rules:
        - back() replays the top song and removes it from history.
        - Playing from history does NOT move the playlist position.
        """
        if self.history.is_empty():
            return None

        replayed_song = self.history.pop()
        self.currently_playing = replayed_song
        # Playlist position is NOT moved!
        return self.currently_playing


### Demo 1: Up Next Priority & Playlist Position Freezing
Checking that songs in Up Next take priority, and the playlist position stays completely frozen while Up Next plays.


In [2]:
album = Playlist("Starboy Album")
p1 = Song("P1", "Starboy", "The Weeknd")
p2 = Song("P2", "Party Monster", "The Weeknd")
p3 = Song("P3", "False Alarm", "The Weeknd")
album.addSong(p1)
album.addSong(p2)
album.addSong(p3)

player = TuneWavePlayer(album)

# Extra tracks for Up Next
u1 = Song("U1", "Levitating", "Dua Lipa")
u2 = Song("U2", "Save Your Tears", "The Weeknd")
player.catalog["U1"] = u1
player.catalog["U2"] = u2

print("Initial song playing:", player.currently_playing)
print("Playlist cursor is on:", player.playlist.getCurrentSong())

# Advance once in playlist
print("\n-- Advancing to P2 --")
print("Playing:", player.playNext())
print("Playlist cursor is now:", player.playlist.getCurrentSong())

# Now queue U1 and U2
print("\n-- Adding U1 and U2 to Up Next --")
player.addToUpNext("U1")
player.addToUpNext("U2")

# Play Next -> Should play U1, keeping playlist cursor frozen on P2!
print("\n-- Calling playNext() (should play U1) --")
print("Playing:", player.playNext())
print("Playlist cursor (frozen!):", player.playlist.getCurrentSong())

# Play Next -> Should play U2, keeping playlist cursor frozen on P2!
print("\n-- Calling playNext() (should play U2) --")
print("Playing:", player.playNext())
print("Playlist cursor (still frozen!):", player.playlist.getCurrentSong())

# Play Next -> Up Next is now empty, resumes playlist at P3!
print("\n-- Calling playNext() (Up Next empty -> resumes playlist at P3) --")
print("Playing:", player.playNext())
print("Playlist cursor:", player.playlist.getCurrentSong())


Initial song playing: Song(P1, 'Starboy' - The Weeknd)
Playlist cursor is on: Song(P1, 'Starboy' - The Weeknd)

-- Advancing to P2 --
Playing: Song(P2, 'Party Monster' - The Weeknd)
Playlist cursor is now: Song(P2, 'Party Monster' - The Weeknd)

-- Adding U1 and U2 to Up Next --

-- Calling playNext() (should play U1) --
Playing: Song(U1, 'Levitating' - Dua Lipa)
Playlist cursor (frozen!): Song(P2, 'Party Monster' - The Weeknd)

-- Calling playNext() (should play U2) --
Playing: Song(U2, 'Save Your Tears' - The Weeknd)
Playlist cursor (still frozen!): Song(P2, 'Party Monster' - The Weeknd)

-- Calling playNext() (Up Next empty -> resumes playlist at P3) --
Playing: Song(P3, 'False Alarm' - The Weeknd)
Playlist cursor: Song(P3, 'False Alarm' - The Weeknd)


### Demo 2: Rewinding with `back()`
Testing `back()` popping from history and verifying the playlist cursor does not move.


In [3]:
print("Currently playing:", player.currently_playing)
print("Playlist cursor is at:", player.playlist.getCurrentSong())

# 1st back -> returns U2
print("\n1st back():", player.back())
print("Playlist cursor after back (must still be P3):", player.playlist.getCurrentSong())

# 2nd back -> returns U1
print("\n2nd back():", player.back())
print("Playlist cursor after back:", player.playlist.getCurrentSong())

# 3rd back -> returns P2
print("\n3rd back():", player.back())
print("Playlist cursor after back:", player.playlist.getCurrentSong())


Currently playing: Song(P3, 'False Alarm' - The Weeknd)
Playlist cursor is at: Song(P3, 'False Alarm' - The Weeknd)

1st back(): Song(U2, 'Save Your Tears' - The Weeknd)
Playlist cursor after back (must still be P3): Song(P3, 'False Alarm' - The Weeknd)

2nd back(): Song(U1, 'Levitating' - Dua Lipa)
Playlist cursor after back: Song(P3, 'False Alarm' - The Weeknd)

3rd back(): Song(P2, 'Party Monster' - The Weeknd)
Playlist cursor after back: Song(P3, 'False Alarm' - The Weeknd)


### Demo 3: Bounded History (Capacity = 10)
Verifying that pushing more than 10 tracks drops the oldest track in $O(1)$.


In [4]:
hist = PlayHistory(capacity=10)

# Simulate playing 15 tracks
for i in range(1, 16):
    hist.push(Song(f"T{i}", f"Track {i}", "Artist"))

print("Total pushed: 15")
print("Current history length:", len(hist), "(must be 10)")
print("Oldest song in history:", hist.to_list()[0])
print("Newest song at top of history:", hist.peek())


Total pushed: 15
Current history length: 10 (must be 10)
Oldest song in history: Song(T6, 'Track 6' - Artist)
Newest song at top of history: Song(T15, 'Track 15' - Artist)


### Sanity Tests for Milestone 2


In [5]:
def run_m2_tests():
    pl = Playlist("Test")
    s1 = Song("S1", "Song 1", "Artist")
    s2 = Song("S2", "Song 2", "Artist")
    s3 = Song("S3", "Song 3", "Artist")
    pl.addSong(s1)
    pl.addSong(s2)
    pl.addSong(s3)

    player = TuneWavePlayer(pl)
    extra = Song("E1", "Extra", "Artist")
    player.catalog["E1"] = extra

    # 1. Start on S1
    assert player.currently_playing == s1
    assert player.playlist.getCurrentSong() == s1

    # 2. Advance to S2
    assert player.playNext() == s2
    assert player.playlist.getCurrentSong() == s2
    assert player.history.peek() == s1

    # 3. Add to Up Next
    player.addToUpNext("E1")
    assert player.playNext() == extra
    # Playlist cursor MUST not move!
    assert player.playlist.getCurrentSong() == s2
    assert player.history.peek() == s2

    # 4. Resume playlist at S3
    assert player.playNext() == s3
    assert player.playlist.getCurrentSong() == s3

    # 5. Back button
    assert player.back() == extra
    assert player.playlist.getCurrentSong() == s3  # frozen!
    assert player.back() == s2
    assert player.playlist.getCurrentSong() == s3  # frozen!

    print("All Milestone 2 tests passed successfully!")

run_m2_tests()


All Milestone 2 tests passed successfully!
